<a href="https://colab.research.google.com/github/Al3xMR/RecuperacionDeInformacion2025B/blob/main/10reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [4]:
#!pip install beir

In [5]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

In [ ]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

In [8]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [9]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [10]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [11]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [ ]:
#!pip install rank-bm25
#!pip install scikit-learn

In [13]:
import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import ndcg_score
import re
from typing import List, Dict, Tuple

In [14]:
# Preprocesamiento de texto
def preprocess_text(text: str) -> List[str]:
    """Tokeniza y preprocesa el texto para BM25"""
    # Convertir a minúsculas
    text = text.lower()
    # Remover caracteres especiales y números
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenizar
    tokens = text.split()
    # Remover stopwords
    tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    return tokens

In [15]:
# Preparar corpus para BM25
print("Preprocesando corpus...")
corpus_tokens = []
doc_ids = []
for doc_id, doc in corpus.items():
    # Combinar título y texto para mejor representación
    text = f"{doc['title']} {doc['text']}"
    tokens = preprocess_text(text)
    corpus_tokens.append(tokens)
    doc_ids.append(doc_id)

Preprocesando corpus...


In [16]:
# Crear índice BM25
print("Creando índice BM25...")
bm25 = BM25Okapi(corpus_tokens)

Creando índice BM25...


In [17]:
# Función para realizar retrieval BM25
def bm25_retrieval(query: str, bm25_model, doc_ids_list: List[str], k: int = 100) -> List[Tuple[str, float]]:
    """Realiza retrieval BM25 para una query"""
    query_tokens = preprocess_text(query)
    scores = bm25_model.get_scores(query_tokens)

    # Obtener top-k documentos
    top_indices = np.argsort(scores)[::-1][:k]
    results = [(doc_ids_list[idx], scores[idx]) for idx in top_indices]
    return results

In [18]:
# Función para evaluar métricas
def evaluate_retrieval(retrieval_results: Dict[str, List[Tuple[str, float]]],
                      qrels_dict: Dict[str, Dict[str, int]],
                      k: int = 10) -> Dict[str, float]:
    """Evalúa Recall@k y nDCG@k para todas las queries"""

    recall_scores = []
    ndcg_scores = []

    for qid, results in retrieval_results.items():
        # Obtener documentos relevantes para esta query
        relevant_docs = qrels_dict.get(qid, {})

        if not relevant_docs:
            continue

        # Extraer IDs de documentos recuperados
        retrieved_docs = [doc_id for doc_id, _ in results[:k]]

        # Calcular Recall@k
        relevant_retrieved = sum(1 for doc_id in retrieved_docs if doc_id in relevant_docs)
        total_relevant = len(relevant_docs)
        recall_at_k = relevant_retrieved / total_relevant if total_relevant > 0 else 0
        recall_scores.append(recall_at_k)

        # Calcular nDCG@k
        # Crear array de relevancias ideales (ordenadas por relevancia descendente)
        ideal_relevances = sorted(relevant_docs.values(), reverse=True)[:k]
        ideal_relevances = ideal_relevances + [0] * (k - len(ideal_relevances))

        # Crear array de relevancias obtenidas
        actual_relevances = []
        for doc_id in retrieved_docs:
            relevance = relevant_docs.get(doc_id, 0)
            actual_relevances.append(relevance)

        # Rellenar con ceros si hay menos de k documentos
        actual_relevances = actual_relevances + [0] * (k - len(actual_relevances))

        # Calcular nDCG@k
        ndcg_at_k = ndcg_score([ideal_relevances], [actual_relevances], k=k)
        ndcg_scores.append(ndcg_at_k)

    # Calcular promedios
    avg_recall = np.mean(recall_scores)
    avg_ndcg = np.mean(ndcg_scores)

    return {
        f"Recall@{k}": avg_recall,
        f"nDCG@{k}": avg_ndcg,
        "num_queries_evaluated": len(recall_scores)
    }

In [19]:
# Realizar retrieval para todas las queries
print("\nRealizando retrieval BM25 para todas las queries...")
bm25_results = {}

for qid, query_text in queries.items():
    results = bm25_retrieval(query_text, bm25, doc_ids, k=100)
    bm25_results[qid] = results


Realizando retrieval BM25 para todas las queries...


In [20]:
# Evaluar métricas
print("\nEvaluando métricas del baseline BM25...")
metrics = evaluate_retrieval(bm25_results, qrels, k=10)

print(f"\n=== Resultados Baseline BM25 ===")
print(f"Recall@10: {metrics['Recall@10']:.4f}")
print(f"nDCG@10: {metrics['nDCG@10']:.4f}")
print(f"Número de queries evaluadas: {metrics['num_queries_evaluated']}")


Evaluando métricas del baseline BM25...

=== Resultados Baseline BM25 ===
Recall@10: 0.7745
nDCG@10: 0.7293
Número de queries evaluadas: 300


In [21]:
# También podemos mostrar resultados para la query de ejemplo
print(f"\n=== Resultados para query ejemplo (ID: {qid}) ===")
query_example = queries[qid]
print(f"Query: {query_example}")

top_10_bm25 = bm25_results[qid][:10]
print(f"\nTop 10 documentos BM25:")
for i, (doc_id, score) in enumerate(top_10_bm25, 1):
    relevance = qrels.get(qid, {}).get(doc_id, 0)
    doc_title = corpus[doc_id]['title'][:50] + "..." if len(corpus[doc_id]['title']) > 50 else corpus[doc_id]['title']
    print(f"{i}. Doc ID: {doc_id} | Score: {score:.4f} | Relevancia: {relevance} | Título: {doc_title}")


=== Resultados para query ejemplo (ID: 1395) ===
Query: p16INK4A accumulation is  linked to an abnormal wound response caused by the microinvasive step of advanced Oral Potentially Malignant Lesions (OPMLs).

Top 10 documentos BM25:
1. Doc ID: 13923069 | Score: 16.4930 | Relevancia: 0 | Título: Targeted nanoparticles containing the proresolving...
2. Doc ID: 17717391 | Score: 15.2872 | Relevancia: 1 | Título: Monitoring Tumorigenesis and Senescence In Vivo wi...
3. Doc ID: 16627684 | Score: 14.4802 | Relevancia: 0 | Título: Hmga2 Promotes Neural Stem Cell Self-Renewal in Yo...
4. Doc ID: 19343151 | Score: 13.2735 | Relevancia: 0 | Título: p16INK4A is a robust in vivo biomarker of cellular...
5. Doc ID: 10698739 | Score: 13.2488 | Relevancia: 0 | Título: Modulation of mitochondrial function and morpholog...
6. Doc ID: 15879931 | Score: 13.1877 | Relevancia: 0 | Título: Regulated Accumulation of Desmosterol Integrates M...
7. Doc ID: 18218379 | Score: 12.8501 | Relevancia: 0 | Título: Q

In [22]:
# Calcular precisión y recall para la query de ejemplo
relevant_docs_example = set(qrels[qid].keys())
retrieved_docs_example = [doc_id for doc_id, _ in top_10_bm25]

precision_at_10 = sum(1 for doc_id in retrieved_docs_example if doc_id in relevant_docs_example) / 10
recall_at_10 = sum(1 for doc_id in retrieved_docs_example if doc_id in relevant_docs_example) / len(relevant_docs_example)

print(f"\nMétricas para query ejemplo:")
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")


Métricas para query ejemplo:
Precision@10: 0.1000
Recall@10: 1.0000


In [23]:
# Guardar resultados para uso posterior
import pickle

results_dir = "../results"
import os
os.makedirs(results_dir, exist_ok=True)

with open(f"{results_dir}/bm25_results.pkl", "wb") as f:
    pickle.dump({
        'bm25_results': bm25_results,
        'metrics': metrics,
        'corpus_tokens': corpus_tokens,
        'doc_ids': doc_ids
    }, f)

print(f"\nResultados guardados en: {results_dir}/bm25_results.pkl")



Resultados guardados en: ../results/bm25_results.pkl


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [ ]:
#!pip install sentence-transformers

In [25]:
from sentence_transformers import CrossEncoder
import torch

In [26]:
# Verificamos si hay GPU disponible para acelerar el proceso
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilizando dispositivo: {device}")

Utilizando dispositivo: cuda


In [ ]:
# 2. Cargar el modelo Cross-Encoder pre-entrenado
# Usamos uno ligero y eficiente entrenado en MS MARCO
model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
cross_encoder = CrossEncoder(model_name, device=device)

In [28]:
# 3. Función de Re-ranking
def rerank_results(query_text, initial_results, corpus, top_k_rerank=50):
    """
    Toma los resultados iniciales (BM25), prepara pares (query, doc)
    y los re-ordena usando el Cross-Encoder.
    """
    # Tomamos solo los top-k candidatos del retrieval inicial para no saturar
    candidates = initial_results[:top_k_rerank]

    # Preparamos los pares [Query, Document Text]
    # Nota: El modelo espera una lista de listas: [['query', 'doc1'], ['query', 'doc2']]
    model_inputs = []
    candidate_ids = []

    for doc_id, _ in candidates:
        doc_text = f"{corpus[doc_id]['title']} {corpus[doc_id]['text']}"
        model_inputs.append([query_text, doc_text])
        candidate_ids.append(doc_id)

    if not model_inputs:
        return []

    # Predecir scores (esto devuelve un array de floats)
    cross_scores = cross_encoder.predict(model_inputs)

    # Empaquetar IDs con sus nuevos scores y ordenar descendente
    reranked_results = list(zip(candidate_ids, cross_scores))
    reranked_results = sorted(reranked_results, key=lambda x: x[1], reverse=True)

    return reranked_results

In [29]:
# 4. Ejecutar el re-ranking para todas las queries
print(f"Iniciando re-ranking Cross-Encoder sobre los resultados de BM25...")
ce_results = {}

# Definimos cuántos documentos de BM25 vamos a re-evaluar
TOP_K_RERANK = 50

for qid in bm25_results:
    query_text = queries[qid]
    initial_hits = bm25_results[qid]

    # Aplicar re-ranking
    reranked = rerank_results(query_text, initial_hits, corpus, top_k_rerank=TOP_K_RERANK)
    ce_results[qid] = reranked

print("Re-ranking completado.")

Iniciando re-ranking Cross-Encoder sobre los resultados de BM25...
Re-ranking completado.


In [30]:
# 5. Análisis: Identificar cambios de posición en el Top 10
# Usaremos la misma query de ejemplo (qid = "133") definida en la Parte 1
qid_example = "133"

print(f"\n=== Análisis de cambios para Query ID: {qid_example} ===")
print(f"Query: {queries[qid_example]}")

# Obtener rankings
bm25_top10_ids = [doc_id for doc_id, _ in bm25_results[qid_example][:10]]
ce_top10_ids = [doc_id for doc_id, _ in ce_results[qid_example][:10]]

print(f"\n{'Pos':<4} | {'BM25 DocID':<15} | {'Cross-Encoder DocID':<20} | {'Cambio?'}")
print("-" * 60)

for i in range(10):
    bm25_doc = bm25_top10_ids[i] if i < len(bm25_top10_ids) else "-"
    ce_doc = ce_top10_ids[i] if i < len(ce_top10_ids) else "-"

    # Verificar si el documento que está ahora en la posición i es nuevo en el top 10
    # o si es el mismo que antes
    status = ""
    if ce_doc == bm25_doc:
        status = "Igual"
    elif ce_doc in bm25_top10_ids:
        old_pos = bm25_top10_ids.index(ce_doc)
        status = f"Subió desde pos {old_pos+1}"
    else:
        status = "Nuevo en Top 10 (estaba > 10 en BM25)"

    print(f"{i+1:<4} | {bm25_doc:<15} | {ce_doc:<20} | {status}")


=== Análisis de cambios para Query ID: 133 ===
Query: Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Pos  | BM25 DocID      | Cross-Encoder DocID  | Cambio?
------------------------------------------------------------
1    | 5270265         | 35660758             | Nuevo en Top 10 (estaba > 10 en BM25)
2    | 45764440        | 12640810             | Subió desde pos 6
3    | 26688294        | 16280642             | Nuevo en Top 10 (estaba > 10 en BM25)
4    | 12785130        | 36345185             | Nuevo en Top 10 (estaba > 10 en BM25)
5    | 37964706        | 6969753              | Subió desde pos 10
6    | 12640810        | 9507605              | Subió desde pos 7
7    | 9507605         | 86694016             | Nuevo en Top 10 (estaba > 10 en BM25)
8    | 35884026        | 19752008             | Nuevo en Top 10 (estaba > 10 en BM25)
9    | 11887584        | 17934082         

In [31]:
# Guardar resultados del Cross-Encoder
import pickle
with open(f"{results_dir}/ce_results.pkl", "wb") as f:
    pickle.dump(ce_results, f)
print(f"\nResultados CE guardados en: {results_dir}/ce_results.pkl")


Resultados CE guardados en: ../results/ce_results.pkl


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [32]:
!pip install lightgbm

In [33]:
import lightgbm as lgb
import numpy as np
from tqdm import tqdm

In [ ]:
# ---------------------------------------------------------
# A. Preparación de Datos de Entrenamiento (Train Set)
# ---------------------------------------------------------
print("Cargando datos de entrenamiento para LTR...")
# Cargamos las queries y qrels de entrenamiento (usamos el mismo corpus)
_, queries_train, qrels_train = GenericDataLoader(dataset_path).load(split="train")

In [35]:
# ---------------------------------------------------------
# B. Ingeniería de Características (Feature Engineering)
# ---------------------------------------------------------
def extract_features(query_text, doc_text, bm25_score):
    """
    Extrae un vector de características simple para un par (query, doc).
    """
    q_tokens = set(preprocess_text(query_text))
    d_tokens = set(preprocess_text(doc_text))

    # Feature 1: BM25 Score (ya lo tenemos)
    f_bm25 = bm25_score

    # Feature 2: Longitud del documento
    f_doc_len = len(d_tokens)

    # Feature 3: Longitud de la query
    f_query_len = len(q_tokens)

    # Feature 4: Jaccard Similarity (solapamiento de términos)
    intersection = len(q_tokens.intersection(d_tokens))
    union = len(q_tokens.union(d_tokens))
    f_jaccard = intersection / union if union > 0 else 0

    # Feature 5: Exact Match en título (binario)
    # Asumimos que doc_text empieza con el título.
    # Para simplificar, verificamos si la query está contenida tal cual en el doc.
    f_exact = 1.0 if query_text.lower() in doc_text.lower() else 0.0

    return [f_bm25, f_doc_len, f_query_len, f_jaccard, f_exact]

In [ ]:
# ---------------------------------------------------------
# C. Construcción del Dataset de Entrenamiento
# ---------------------------------------------------------
print("Generando dataset de entrenamiento (mining negatives)...")

X_train = []
y_train = []
groups_train = [] # LightGBM Ranker necesita saber cuántos docs hay por query

# Para entrenar, necesitamos ejemplos positivos y negativos.
# Usaremos BM25 para minar negativos "difíciles" del set de train.
for qid, query_text in tqdm(queries_train.items(), desc="Processing Train Queries"):
    if qid not in qrels_train:
        continue

    # 1. Obtener candidatos con BM25 (top 10 para entrenar rápido)
    # Nota: Usamos el índice BM25 que ya creamos en Parte 2 sobre el corpus completo
    hits = bm25_retrieval(query_text, bm25, doc_ids, k=10)

    current_group_count = 0
    relevant_docs = qrels_train[qid]

    for doc_id, score in hits:
        doc_obj = corpus[doc_id]
        doc_text = f"{doc_obj['title']} {doc_obj['text']}"

        # Extraer features
        feats = extract_features(query_text, doc_text, score)

        # Determinar etiqueta (Label): 1 si es relevante, 0 si no
        label = 1 if doc_id in relevant_docs else 0

        X_train.append(feats)
        y_train.append(label)
        current_group_count += 1

    if current_group_count > 0:
        groups_train.append(current_group_count)

X_train = np.array(X_train)
y_train = np.array(y_train)

In [38]:
# ---------------------------------------------------------
# D. Entrenamiento del Modelo LTR (LambdaRank)
# ---------------------------------------------------------
print(f"\nEntrenando LightGBM Ranker con {len(X_train)} muestras...")

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=100,
    learning_rate=0.1
)

# Eliminamos 'verbose=False' para evitar el TypeError
ranker.fit(
    X_train,
    y_train,
    group=groups_train
)
print("Modelo entrenado.")


Entrenando LightGBM Ranker con 8090 muestras...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002290 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 675
[LightGBM] [Info] Number of data points in the train set: 8090, number of used features: 4
Modelo entrenado.


In [ ]:
# ---------------------------------------------------------
# E. Re-ranking del Test Set (Inferencia)
# ---------------------------------------------------------
print("\nAplicando modelo LTR a los resultados de test (BM25)...")

ltr_results = {}
TOP_K_RERANK_LTR = 50 # Re-rankeamos el top 50 de BM25

for qid in bm25_results:
    query_text = queries[qid]
    initial_hits = bm25_results[qid][:TOP_K_RERANK_LTR]

    if not initial_hits:
        ltr_results[qid] = []
        continue

    # Preparar batch de features para esta query
    X_test_batch = []
    candidates_ids = []

    for doc_id, score in initial_hits:
        doc_obj = corpus[doc_id]
        doc_text = f"{doc_obj['title']} {doc_obj['text']}"
        feats = extract_features(query_text, doc_text, score)

        X_test_batch.append(feats)
        candidates_ids.append(doc_id)

    # Predecir scores
    ltr_scores = ranker.predict(np.array(X_test_batch))

    # Ordenar
    ranked_candidates = list(zip(candidates_ids, ltr_scores))
    ranked_candidates = sorted(ranked_candidates, key=lambda x: x[1], reverse=True)

    ltr_results[qid] = ranked_candidates

In [40]:
# ---------------------------------------------------------
# F. Análisis de cambios (Igual que parte 3)
# ---------------------------------------------------------
print(f"\n=== Análisis de cambios (LTR) para Query ID: {qid_example} ===")
ltr_top10_ids = [doc_id for doc_id, _ in ltr_results[qid_example][:10]]

print(f"{'Pos':<4} | {'BM25 DocID':<15} | {'LTR DocID':<20} | {'Cambio?'}")
print("-" * 60)

for i in range(10):
    bm25_doc = bm25_top10_ids[i] if i < len(bm25_top10_ids) else "-"
    ltr_doc = ltr_top10_ids[i] if i < len(ltr_top10_ids) else "-"

    status = ""
    if ltr_doc == bm25_doc:
        status = "Igual"
    elif ltr_doc in bm25_top10_ids:
        old_pos = bm25_top10_ids.index(ltr_doc)
        status = f"Subió desde pos {old_pos+1}"
    else:
        status = "Nuevo en Top 10"

    print(f"{i+1:<4} | {bm25_doc:<15} | {ltr_doc:<20} | {status}")

# Guardar resultados
with open(f"{results_dir}/ltr_results.pkl", "wb") as f:
    pickle.dump(ltr_results, f)


=== Análisis de cambios (LTR) para Query ID: 133 ===
Pos  | BM25 DocID      | LTR DocID            | Cambio?
------------------------------------------------------------
1    | 5270265         | 5270265              | Igual
2    | 45764440        | 12640810             | Subió desde pos 6
3    | 26688294        | 26688294             | Igual
4    | 12785130        | 45764440             | Subió desde pos 2
5    | 37964706        | 35884026             | Subió desde pos 8
6    | 12640810        | 11887584             | Subió desde pos 9
7    | 9507605         | 86694016             | Nuevo en Top 10
8    | 35884026        | 23260700             | Nuevo en Top 10
9    | 11887584        | 12785130             | Subió desde pos 4
10   | 6969753         | 9507605              | Subió desde pos 7


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [42]:
import pandas as pd
import numpy as np
from sklearn.metrics import ndcg_score

In [43]:
def compute_metrics(results_dict, qrels_dict, k=10):
    """
    Calcula nDCG@k, Recall@k y MAP@k para un conjunto de resultados.
    """
    ndcg_scores = []
    recall_scores = []
    ap_scores = [] # Average Precision scores

    for qid, results in results_dict.items():
        if qid not in qrels_dict:
            continue

        relevant_docs = qrels_dict[qid] # Diccionario {doc_id: relevance}
        if not relevant_docs:
            continue

        # Tomamos solo los top-k resultados
        top_hits = results[:k]

        # --- Cálculo de Recall@k ---
        retrieved_ids = [doc_id for doc_id, _ in top_hits]
        relevant_retrieved = sum(1 for doc in retrieved_ids if doc in relevant_docs)
        total_relevant = len(relevant_docs)
        recall_scores.append(relevant_retrieved / total_relevant if total_relevant > 0 else 0)

        # --- Cálculo de nDCG@k ---
        # Relevancias ideales (ordenadas desc)
        ideal_rels = sorted(relevant_docs.values(), reverse=True)[:k]
        ideal_rels += [0] * (k - len(ideal_rels))

        # Relevancias obtenidas
        actual_rels = [relevant_docs.get(doc_id, 0) for doc_id in retrieved_ids]
        actual_rels += [0] * (k - len(actual_rels))

        ndcg_scores.append(ndcg_score([ideal_rels], [actual_rels], k=k))

        # --- Cálculo de MAP@k (Mean Average Precision) ---
        # AP = Promedio de la precisión en cada posición donde recuperamos un doc relevante
        hits = 0
        sum_precisions = 0

        for i, doc_id in enumerate(retrieved_ids):
            if doc_id in relevant_docs:
                hits += 1
                precision_at_i = hits / (i + 1)
                sum_precisions += precision_at_i

        # MAP divide por el número total de relevantes (o k, dependiendo de la definición estricta)
        # Aquí usamos la definición estándar: dividir por min(len(relevant), k) o total_relevant
        # Para ser consistentes en retrieval se suele dividir por el total de relevantes conocidos.
        ap = sum_precisions / total_relevant if total_relevant > 0 else 0
        ap_scores.append(ap)

    return {
        "nDCG@10": np.mean(ndcg_scores),
        "Recall@10": np.mean(recall_scores),
        "MAP@10": np.mean(ap_scores)
    }

In [44]:
print("Evaluando resultados finales...")

# 1. Evaluar Baseline
metrics_bm25 = compute_metrics(bm25_results, qrels, k=10)

# 2. Evaluar Cross-Encoder
metrics_ce = compute_metrics(ce_results, qrels, k=10)

# 3. Evaluar LTR
metrics_ltr = compute_metrics(ltr_results, qrels, k=10)

# 4. Crear Tabla Comparativa
df_results = pd.DataFrame([
    metrics_bm25,
    metrics_ltr,
    metrics_ce
], index=["BM25 (Baseline)", "LTR (LightGBM)", "Cross-Encoder"])

Evaluando resultados finales...


In [ ]:
# Mostrar resultados
print("\n" + "="*40)
print("TABLA COMPARATIVA FINAL (SCIFACT TEST)")
print("="*40)
print(df_results.round(4))


TABLA COMPARATIVA FINAL (SCIFACT TEST)
                 nDCG@10  Recall@10  MAP@10
BM25 (Baseline)   0.7293     0.7745  0.6070
LTR (LightGBM)    0.7779     0.7651  0.6558
Cross-Encoder     0.7560     0.8018  0.6442
